# 03. GroupBy — продвинутые техники

В `02_processing_data.ipynb` (секция 11) уже разобраны:
- базовый `groupby` + `.agg()` по одному ключу
- `transform` для добавления агрегата обратно в df

Здесь — всё остальное из `pandas_groupby.py`:
1. Multi-key groupby — итерация и dict-доступ
2. Groupby по mapping колонок (группировка самих колонок)
3. Кастомные функции в `.agg()` — одна, список, по-колонке
4. `pd.cut` / `pd.qcut` + groupby для биннинга
5. `.apply()` — паттерн SEPARATE-APPLY-COMBINE
6. `transform` vs `apply` — в чём разница и когда что

## 1. Multi-key groupby — итерация и dict-доступ

Группировать можно по нескольким ключам сразу.
Результирующий индекс — **MultiIndex** (иерархический).

Итерация: `for (k1, k2), group in df.groupby([...])`
Dict-доступ: превратить в словарь `{(k1,k2): sub_df}` и обращаться по ключу.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Исторические сигналы от разных моделей по разным тикерам
signals = pd.DataFrame({
    'ticker': ['AAPL', 'AAPL', 'GOOG', 'GOOG', 'MSFT', 'MSFT', 'AAPL'],
    'model':  ['ModelA', 'ModelB', 'ModelA', 'ModelB', 'ModelA', 'ModelB', 'ModelA'],
    'score':  np.round(np.random.uniform(0, 1, 7), 2),   # уверенность модели
    'pnl':   np.round(np.random.normal(0, 0.5, 7), 3),  # P&L сигнала
})

print('Исходные сигналы:')
print(signals)

# Группировка по двум ключам: тикер + модель
grouped = signals.groupby(['ticker', 'model'])

# Агрегация — результат с MultiIndex
print('\nСредний score и суммарный P&L (ticker + model):')
print(grouped[['score', 'pnl']].agg({'score': 'mean', 'pnl': 'sum'}))

In [ ]:
# Итерация по группам — имя группы это кортеж ключей
print('Итерация по группам (ticker, model):')
for (ticker, model), group in signals.groupby(['ticker', 'model']):
    avg_score = group['score'].mean()
    print(f'  {ticker} / {model}: avg_score={avg_score:.3f}, n={len(group)}')

# Dict-доступ — удобно когда нужна одна конкретная группа
groups_dict = {name: grp for name, grp in signals.groupby(['ticker', 'model'])}
print('\nДанные только AAPL / ModelA:')
print(groups_dict[('AAPL', 'ModelA')])

## 2. Groupby по mapping колонок (группировка самих колонок)

Обычно `groupby` группирует **строки** по значениям в колонках.
Но можно группировать сами **колонки** — передав словарь `{col_name: group_name}`.

Зачем: когда у тебя много однотипных колонок (напр. цены по разным биржам / категориям),
и нужно считать агрегат по группам колонок.

In [ ]:
# Матрица фичей: для каждого актива — несколько колонок разных категорий
# price_* — ценовые фичи, risk_* — рисковые, volume_* — объёмные
np.random.seed(0)
assets = ['AAPL', 'GOOG', 'MSFT', 'JPM', 'XOM']
feature_matrix = pd.DataFrame(
    np.round(np.random.randn(5, 6), 3),
    index=assets,
    columns=['price_close', 'price_ma20', 'risk_beta', 'risk_vol', 'volume_avg', 'volume_rel']
)
print('Матрица фичей:')
print(feature_matrix)

# Маппинг: каждая колонка → группа
col_mapping = {
    'price_close': 'price',
    'price_ma20':  'price',
    'risk_beta':   'risk',
    'risk_vol':    'risk',
    'volume_avg':  'volume',
    'volume_rel':  'volume',
}

# Группируем колонки и суммируем — получаем агрегат по группам колонок
# axis=1 означает: группировать по колонкам (не по строкам)
by_feature_group = feature_matrix.groupby(col_mapping, axis=1)
print('\nСреднее по группам колонок (price / risk / volume):')
print(by_feature_group.mean())

## 3. Кастомные функции в `.agg()`

`.agg()` принимает:
- строку: `'mean'`, `'std'`, `'count'` и т.д.
- свою функцию: `agg(my_func)`
- список функций: `agg(['mean', my_func])`
- словарь — разные функции для разных колонок: `agg({'col1': ['mean'], 'col2': my_func})`

Функция принимает **Series** (одна колонка в группе) и возвращает скаляр.

In [ ]:
np.random.seed(7)

# Дневная доходность 3 активов за 60 торговых дней
dates = pd.date_range('2024-01-01', periods=60, freq='B')  # B = рабочие дни
tickers = ['AAPL', 'GOOG', 'MSFT']
returns_df = pd.DataFrame(
    np.round(np.random.normal(0.001, 0.02, (60, 3)), 4),
    index=dates,
    columns=tickers
)
# Добавим колонку месяца для группировки
returns_df['month'] = returns_df.index.month

grouped = returns_df.groupby('month')

# Кастомная функция: размах (max - min)
def range_spread(series):
    """Разница max-min — мера волатильности внутри группы"""
    return series.max() - series.min()

# 1. Одна кастомная функция для всех числовых колонок
print('Размах доходности по месяцам (max-min):')
print(grouped[['AAPL', 'GOOG', 'MSFT']].agg(range_spread)) # or grouped.agg(range_spread) — если хотим все колонки, включая month

In [ ]:
# 2. Список функций — несколько метрик за раз (MultiIndex в колонках)
print('Несколько метрик за раз для AAPL:')
print(grouped['AAPL'].agg(['mean', 'std', range_spread]))

# 3. Разные функции для разных колонок
print('\nРазные агрегации для каждого тикера:')
result = grouped[['AAPL', 'GOOG', 'MSFT']].agg({
    'AAPL': ['mean', 'std'],           # ср.доходность и волатильность
    'GOOG': range_spread,              # кастомный размах
    'MSFT': ['min', 'max', 'count'],   # экстремумы и кол-во дней
})
print(result)

## 4. `pd.cut` / `pd.qcut` + groupby для биннинга

Паттерн: разбить числовой признак на бины → использовать как ключ группировки.

- `pd.cut(x, n)` — равные интервалы по значению
- `pd.qcut(x, n)` — равное количество наблюдений в каждом бине (квантили)

После этого `groupby(bins).agg(...)` считает статистику внутри каждого бина.
Применяется для: анализа профиля по P/E-квинтилям, по объёмным уровням, и т.д.

In [ ]:
np.random.seed(99)
n = 200

# Симуляция скринера акций: P/E ratio и 1-летняя доходность
screener = pd.DataFrame({
    'ticker':    [f'STK{i:03d}' for i in range(n)],
    'pe_ratio':  np.abs(np.random.normal(20, 10, n)).round(1),  # P/E: > 0
    'return_1y': np.random.normal(0.08, 0.25, n).round(4),      # годовая доходность
    'volume_m':  np.abs(np.random.normal(50, 30, n)).round(1),  # объём торгов, млн
})

print(screener.head())

# pd.cut: 4 равных интервала P/E (не равные по кол-ву наблюдений)
screener['pe_bin'] = pd.cut(screener['pe_ratio'], bins=4, labels=['Low','Mid-Low','Mid-High','High'])

# pd.qcut: квартили по объёму (равное кол-во акций в каждом квартиле)
screener['volume_quartile'] = pd.qcut(screener['volume_m'], q=4, labels=['Q1_low','Q2','Q3','Q4_high'])

# Статистика доходности по бинам P/E
print('Доходность по квартилям P/E (cut — равные интервалы):')
print(screener.groupby('pe_bin', observed=True)['return_1y'].agg(['mean','std','count']))

print('\nДоходность по квартилям объёма (qcut — равные размеры групп):')
print(screener.groupby('volume_quartile', observed=True)['return_1y'].agg(['mean','std','count']))

## 5. `.apply()` — паттерн SEPARATE-APPLY-COMBINE

`.apply()` — самый гибкий инструмент groupby.
Функция получает **весь sub-DataFrame** группы (не одну колонку как в `.agg()`).
Это позволяет считать сложные метрики, которые требуют нескольких колонок сразу.

**SEPARATE** → `groupby()` делит df на группы  
**APPLY** → функция считает что-то по каждой группе  
**COMBINE** → pandas собирает результаты обратно

Классический финансовый пример: **взвешенное среднее** (по объёму, по капитализации и т.д.)

In [ ]:
np.random.seed(13)

# Сигналы с весами (чем выше объём — тем больший вес сигнал должен получить)
portfolio = pd.DataFrame({
    'sector':    ['Tech','Tech','Tech','Finance','Finance','Energy','Energy','Energy'],
    'ticker':    ['AAPL','GOOG','MSFT','JPM','GS','XOM','CVX','SLB'],
    'signal':    np.round(np.random.uniform(-1, 1, 8), 3),    # -1..1, торговый сигнал
    'mkt_cap_b': np.round(np.random.uniform(50, 3000, 8), 1), # рыночная кап, млрд
})
print('Портфель:')
print(portfolio)

# SEPARATE-APPLY-COMBINE:
# хотим средний сигнал по сектору, взвешенный по рыночной капитализации

def cap_weighted_signal(group):
    """Взвешенное среднее сигнала по рыночной капитализации"""
    return np.average(group['signal'], weights=group['mkt_cap_b'])

# SEPARATE: groupby разбивает на группы по сектору
# APPLY:    cap_weighted_signal считает по каждой группе
# COMBINE:  pandas собирает Series с результатами
weighted_signals = portfolio.groupby('sector').apply(cap_weighted_signal)
print('\nВзвешенный сигнал по секторам (капитализация = вес):')
print(weighted_signals)

In [ ]:
# apply с функцией, возвращающей DataFrame — COMBINE склеивает по строкам
# Пример: топ-1 актив по сигналу в каждом секторе

def top_signal_asset(group):
    """Возвращает строку с максимальным сигналом в группе"""
    return group.nlargest(1, 'signal')

print('Лучший сигнал в каждом секторе:')
print(portfolio.groupby('sector').apply(top_signal_asset))

## 6. `transform` vs `apply` — в чём разница

| | `.agg()` / `.apply()` | `.transform()` |
|---|---|---|
| Размер результата | меньше исходного (одна строка на группу) | **тот же**, что исходный df |
| Зачем | получить сводную статистику | добавить агрегат как новую колонку |
| Пример | средняя доходность по сектору | добавить столбец «средняя по сектору» к каждой строке |

`transform` — это когда тебе нужно **нормализовать внутри группы** или создать относительный признак.

In [ ]:
np.random.seed(21)

# Ежедневные доходности активов из разных секторов
df = pd.DataFrame({
    'ticker':   ['AAPL','GOOG','MSFT','JPM','GS','XOM','CVX'],
    'sector':   ['Tech','Tech','Tech','Finance','Finance','Energy','Energy'],
    'return_d': np.round(np.random.normal(0.001, 0.015, 7), 4),
})
print('Исходный df:')
print(df)

# --- apply/agg: одна строка на группу ---
sector_avg = df.groupby('sector')['return_d'].mean()
print('\nagg — одно значение на сектор:')
print(sector_avg)
print(f'Строк в результате: {len(sector_avg)} (а не {len(df)})')

# --- transform: результат той же длины что df ---
df['sector_avg_return']  = df.groupby('sector')['return_d'].transform('mean')
df['excess_return']      = df['return_d'] - df['sector_avg_return']  # доходность сверх сектора
df['sector_z_score']     = df.groupby('sector')['return_d'].transform(
    lambda x: (x - x.mean()) / x.std()  # z-score внутри сектора
)

print('\ntransform — агрегат добавлен к каждой строке:')
print(df[['ticker','sector','return_d','sector_avg_return','excess_return','sector_z_score']])

In [ ]:
# Практический вывод: когда что использовать

# groupby + agg  → сводная таблица (одна строка на группу)
# groupby + apply → сложная агрегация с несколькими колонками или нестандартный результат
# groupby + transform → добавить агрегат как фичу к каждой строке

print('Резюме применения:')
summary = pd.DataFrame({
    'метод':     ['agg', 'apply', 'transform'],
    'вход в fn': ['Series (одна колонка)', 'DataFrame (вся группа)', 'Series (одна колонка)'],
    'выход':     ['скаляр → строка на группу', 'скаляр / Series / DataFrame', 'Series той же длины'],
    'зачем':     ['сводная статистика', 'сложная агрегация', 'новая колонка в исходный df'],
})
print(summary.to_string(index=False))